<a href="https://colab.research.google.com/github/jasonkjw/daily_coding_commit/blob/main/climate_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import requests
import pandas as pd
import io
from datetime import datetime, timedelta
import time
from google.colab import drive

# 1. 드라이브 마운트 및 설정
drive.mount('/content/drive')
AUTH_KEY = "FBA6we5nSHuQOsHuZxh7DA"
STATION_ID = "130"
START_YEAR, END_YEAR = 1972, 1972

final_data = []

print("데이터 수집을 시작합니다.")

# 2. 31일 단위로 루프 돌며 원본 호출
curr = datetime(START_YEAR, 6, 1)
end_limit = datetime(END_YEAR, 12, 30, 23)

while curr <= end_limit:
    t1 = curr.strftime("%Y%m%d%H%M")
    t2 = (curr + timedelta(days=30)).strftime("%Y%m%d2300")

    url = f"https://apihub.kma.go.kr/api/typ01/url/kma_sfctm3.php?tm1={t1}&tm2={t2}&stn={STATION_ID}&help=0&authKey={AUTH_KEY}"

    try:
        res = requests.get(url)
        # 공백 구분으로 읽기 (0번: 일시, 11번: TA) [cite: 24, 27]
        df = pd.read_csv(io.StringIO(res.text), sep=r'\s+', comment='#', header=None, on_bad_lines='skip')

        for _, r in df.iterrows():
            tm = str(r[0])
            # 관측소, 년, 월, 일, 시간, 기온 추출
            final_data.append([STATION_ID, tm[:4], tm[4:6], tm[6:8], tm[8:10], r[11]])

    except:
        pass

    curr += timedelta(days=31)
    time.sleep(0.5)

# 3. 요청하신 24시간 가로 포맷으로 정리
df_raw = pd.DataFrame(final_data, columns=['STN', 'Year', 'Month', 'Day', 'Hour', 'TA'])
df_raw['Hour'] = df_raw['Hour'].replace('00', '24') # 00시를 24시로 변경

# 가로로 펼치기 (가장 기본적인 피벗)
result = df_raw.pivot(index=['STN', 'Year', 'Month', 'Day'], columns='Hour', values='TA').reset_index()

# 4. 저장
save_path = "/content/drive/MyDrive/ULJIN_130_FINAL.xlsx"
result.to_excel(save_path, index=False)

print(f"완료되었습니다. 저장 경로: {save_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
데이터 수집을 시작합니다.
완료되었습니다. 저장 경로: /content/drive/MyDrive/ULJIN_130_FINAL.xlsx
